In [ ]:
!pip install -qU langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00


In [ ]:
import requests

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool

from langchain_core.messages import SystemMessage

In [ ]:
# 1. LLM Setup
llm = ChatGroq(model="llama-3.3-70b-versatile",
               temperature=0,
               api_key="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")                # console.groq.com

In [ ]:
# 2. General News Tool (Searchable!)
@tool
def get_news(topic: str) -> str:
    """Fetches the latest news articles about a specific topic."""
    api_key = ""                       # Free from newsapi.org
    url = f"https://newsapi.org/v2/everything?q={topic}&apiKey={api_key}"
    data = requests.get(url).json()
    if data.get("articles"):
        return "\n".join([f"Title: {a['title']}\nContent: {a['content']}" for a in data["articles"][:3]])
    return f"No news found for {topic}."


# 3. Dedicated Summarizer Tool
@tool
def summarize_text(text: str) -> str:
    """Summarizes a long block of text into a concise, easily readable paragraph."""
    # Have the LLM do the heavy lifting inside the tool!
    prompt = f"Summarize this text concisely: {text}"
    return llm.invoke(prompt).content


tools = [get_news, summarize_text]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Worker 1: The Assistant (Brain)
def assistant_node(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Worker 2: The Tool Runner (Hands)
tool_node = ToolNode(tools)

In [ ]:
def should_continue(state: State):
    last_msg = state["messages"][-1]
    if getattr(last_msg, "tool_calls", None):     # LLM wants a tool?
        return "tools"                          # → Go to tool node
    return END                                  # → Done, stop the loop

In [ ]:
builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

In [ ]:
graph = builder.compile()

# Test our News Agent!
sys_msg = SystemMessage(content="You are an AI news summarizer. Use your tools to fetch and summarize news.")

initial_state = {
 "messages": [
    sys_msg,
    ("user", "What is the latest news about AI Agents?")
 ]
}
result = graph.invoke(initial_state)
print(result["messages"][-1].content)

The latest news about AI agents includes Meta's acquisition of Moltbook, a Reddit-like network for AI agents, with the Moltbook team joining Meta's AI division. Furthermore, some tech reporters, such as Alex Heath, are utilizing AI agents to assist with writing and editing their stories, highlighting the growing integration of AI in various industries.
